# Lab 1 · Notebook 3 — Gradio: from a function to a web app ⏱ ~20 min

A model in a notebook is only useful to *you*. **Gradio** turns any Python function into a web interface in a few lines —
perfect for demos, user tests and (soon) your final project.

**Mental model:** `inputs → your function → outputs`. Gradio draws the widgets, calls your function, and renders the result.

**You will:** (1) build a toy interface, (2) put a real model behind it, (3) show named entities with highlighting,
(4) combine several tasks into a tabbed app with `gr.Blocks`, (5) build your own interface.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from functools import lru_cache
import torch, gradio as gr
from transformers import pipeline

print("gradio", gr.__version__)
DEVICE = 0 if torch.cuda.is_available() else -1
SENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
NER_MODEL  = "dslim/distilbert-NER"
FILL_MODEL = "distilbert/distilbert-base-uncased"

@lru_cache(maxsize=None)                      # <- load each model ONCE, then reuse it (very important in apps!)
def get_pipe(task, model, **kwargs):
    return pipeline(task, model=model, device=DEVICE, **kwargs)

> 💡 **Why `lru_cache`?** Loading a model takes seconds. If you loaded it *inside* the function, every click would re-load it.
> Caching makes the first click slower and all the others instant.

## 1 · The smallest possible app

In [ ]:
gr.close_all()                                 # stop any app from an earlier run so the port is free

def greet(name, excited):
    message = f"Hello, {name}!"
    return message.upper() + "!!!" if excited else message

demo = gr.Interface(
    fn=greet,
    inputs=[gr.Textbox(label="Your name"), gr.Checkbox(label="Excited?")],   # one component per function argument
    outputs=gr.Textbox(label="Greeting"),                                     # one component per returned value
)
demo.launch()                                  # in Jupyter the app appears right below this cell

Two lists — `inputs` and `outputs` — matched to the function's arguments and return values. Everything else is optional polish.
Try it, then re-run the cell with a different label to see how the UI follows your code.

## 2 · Put a real model behind it
`gr.Label` is made for classifiers: it takes a **dict `{label: probability}`** and draws confidence bars.

In [ ]:
gr.close_all()

def sentiment(text):
    if not text.strip():
        raise gr.Error("Please type some text first.")             # shows a friendly red message instead of a stack trace
    scores = get_pipe("sentiment-analysis", SENT_MODEL)(text, top_k=None)
    return {s["label"]: s["score"] for s in scores}                 # {"POSITIVE": 0.99, "NEGATIVE": 0.01}

demo = gr.Interface(
    fn=sentiment,
    inputs=gr.Textbox(lines=3, label="Review"),
    outputs=gr.Label(num_top_classes=2, label="Sentiment"),
    title="Sentiment analyzer",
    description="DistilBERT fine-tuned on SST-2 (English only, two labels).",
    examples=["I loved every minute of it!", "The battery died after a week.", "Not bad at all."],
)
demo.launch()

## 3 · Highlight entities in text
`gr.HighlightedText` can colour spans. It accepts `{"text": ..., "entities": [{"entity": label, "start": i, "end": j}, ...]}` —
which is almost exactly what the NER pipeline returns (with `aggregation_strategy="simple"`, character offsets are included).

In [ ]:
gr.close_all()

def find_entities(text):
    entities = get_pipe("token-classification", NER_MODEL, aggregation_strategy="simple")(text)
    return {"text": text,
            "entities": [{"entity": e["entity_group"], "start": e["start"], "end": e["end"]} for e in entities]}

demo = gr.Interface(
    fn=find_entities,
    inputs=gr.Textbox(lines=3, label="Text"),
    outputs=gr.HighlightedText(label="Entities (PER = person, ORG = organisation, LOC = location, MISC)"),
    examples=["Angela Merkel visited the Google office in Zurich last Tuesday.",
              "Hugging Face was founded in New York by Clément Delangue."],
)
demo.launch()

## 4 · Several tasks in one app: `gr.Blocks`
`gr.Interface` is a shortcut for *one function*. For layouts, tabs and several buttons use **`gr.Blocks`**:
you declare components inside `with` blocks and **wire events** explicitly: `button.click(fn, inputs, outputs)`.

In [ ]:
gr.close_all()

with gr.Blocks(title="Mini NLP studio") as studio:
    gr.Markdown("# 🧪 Mini NLP studio\nTwo BERT-family models in one app.")
    with gr.Tab("Sentiment"):
        with gr.Row():
            s_in = gr.Textbox(lines=4, label="Review", scale=2)
            s_out = gr.Label(num_top_classes=2, label="Sentiment", scale=1)
        s_btn = gr.Button("Analyze", variant="primary")
        s_btn.click(sentiment, inputs=s_in, outputs=s_out)           # <- the wiring
        gr.Examples(["Great value for money!", "Awful, never again."], inputs=s_in)
    with gr.Tab("Entities"):
        e_in = gr.Textbox(lines=4, label="Text")
        e_out = gr.HighlightedText(label="Entities")
        gr.Button("Find entities", variant="primary").click(find_entities, e_in, e_out)

studio.launch()

### 🔨 TODO 7 — build your own interface (6 min)
Make a **fill-the-blank** app: the user types a sentence containing `[MASK]`, picks how many guesses to show with a **slider**,
and sees the predictions as a `gr.Label`.

Steps: finish `fill_mask_fn(text, top_k)` (raise `gr.Error` if the mask token is missing), then create `demo_fill = gr.Interface(...)`
with a `gr.Textbox` and a `gr.Slider(1, 10, value=5, step=1)` as inputs.

In [ ]:
def fill_mask_fn(text, top_k):
    fill = get_pipe("fill-mask", FILL_MODEL)
    mask = fill.tokenizer.mask_token
    # TODO 7a: if `mask` is not in `text`, raise gr.Error("...")
    # TODO 7b: results = fill(text, top_k=int(top_k));  return {token_string: float(score), ...}
    raise NotImplementedError("fill me in")

gr.close_all()
# TODO 7c: demo_fill = gr.Interface(fn=..., inputs=[gr.Textbox(...), gr.Slider(1, 10, value=5, step=1, ...)], outputs=gr.Label(...))
# then: demo_fill.launch()

In [ ]:
out = fill_mask_fn("Paris is the [MASK] of France.", 3)
assert len(out) == 3 and "capital" in out, out
raised = False
try:
    fill_mask_fn("There is no blank here.", 3)
except gr.Error:
    raised = True
assert raised, "fill_mask_fn should raise gr.Error when the mask token is missing"
assert isinstance(demo_fill, gr.Interface)
print("✅ TODO 7 passed")

## 5 · Rules of thumb for model apps
| Do | Why |
|---|---|
| Load models **once** (module level / `lru_cache`) | seconds per click otherwise |
| Keep **logic in plain functions**, UI in a thin layer | you can unit-test the functions without a browser |
| **Validate inputs** and raise `gr.Error` | users *will* send empty text or the wrong language |
| Add `examples=[...]` | a demo with no examples is a demo nobody tries |
| Return `{label: prob}` for `gr.Label`, spans for `HighlightedText` | each component has an expected format — read its docs |
| `demo.launch(share=True)` | temporary public link to show your supervisor (don't share private data!) |

**Port already in use?** Run `gr.close_all()` (or restart the kernel). In a terminal, `Ctrl+C`.

## ✅ Checkpoint
* `Interface` = one function, `Blocks` = layout + events;
* why models are loaded once, and what `gr.Error` is for;
* which output component matches which task.

**Next:** Notebook 4 — you assemble everything into **TextLab**, a multi-task app, and export it as a real script.